# 03 -- PD model (probability of default)

**What this notebook does (plain English):** Builds a simple, transparent model
that estimates each loan's **chance of defaulting within one year** from facts
known at the start (credit score, loan-to-value, debt-to-income, loan purpose,
etc.). We use **logistic regression** -- the industry-standard interpretable
scorecard method -- and grade it the way a model-validation team would. The target
is the **one-year** default flag (PD-1/PD-2), the framework's PD basis.

**Headline result:** the model separates good from bad loans well, with an **AUC
around 0.86** (a coin-flip would be 0.50), and its predicted one-year default rates
track the actual ones closely.

In [1]:
import sys, os
ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, 'src')):
    ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import warnings; warnings.filterwarnings('ignore')
print('project root:', ROOT)

project root: D:\Jane\Job Search\Github\bank\github project\freddie mac mortgage


In [2]:
# Load the base table and split into train/test (stratified on default).
import pandas as pd
from sklearn.model_selection import train_test_split
from src import models, metrics
from src.output import save_csv
base = pd.read_parquet('data/processed/analysis_base.parquet')
# PD target = the ONE-YEAR default flag (PD-1/PD-2), not the observed-to-date flag.
PD_TARGET = 'default_within_12m'
train, test = train_test_split(base, test_size=0.30, stratify=base[PD_TARGET], random_state=42)

In [3]:
# Fit the logistic one-year PD on origination features and score the held-out test set.
model, columns = models.fit_pd(train)
test = test.copy()
test['pd_hat'] = models.predict_pd(model, columns, test)

In [4]:
# Grade discrimination (AUC / Gini / KS) on the test set.
y = test[PD_TARGET].astype(int)
auc = metrics.auc(y, test['pd_hat'])
gini = metrics.gini(y, test['pd_hat'])
ks = metrics.ks(y, test['pd_hat'])
print(f'AUC={auc:.3f}  Gini={gini:.3f}  KS={ks:.3f}')

AUC=0.864  Gini=0.728  KS=0.575


In [5]:
# Calibration: do predicted PDs match observed default rates, decile by decile?
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
cal = metrics.calibration_table(y, test['pd_hat'])
ax = cal.plot(x='predicted_pd', y='observed_default_rate', marker='o', legend=False,
              title='PD calibration (predicted vs observed)')
ax.plot([0, cal['predicted_pd'].max()], [0, cal['predicted_pd'].max()], 'k--', lw=1)
ax.set_xlabel('predicted PD'); ax.set_ylabel('observed default rate'); plt.tight_layout()
os.makedirs('output/readme_assets', exist_ok=True)
plt.savefig('output/readme_assets/pd_calibration.png', dpi=110); plt.close()

In [6]:
# Save the metrics + predicted-PD distribution as this notebook's result.
metrics_tbl = pd.DataFrame([
    {'metric': 'AUC', 'value': round(auc, 4)},
    {'metric': 'Gini', 'value': round(gini, 4)},
    {'metric': 'KS', 'value': round(ks, 4)},
    {'metric': 'test_loans', 'value': len(test)},
    {'metric': 'pd_hat_mean', 'value': round(test['pd_hat'].mean(), 4)},
    {'metric': 'pd_hat_p50', 'value': round(test['pd_hat'].median(), 4)},
    {'metric': 'pd_hat_p95', 'value': round(test['pd_hat'].quantile(0.95), 4)},
])
save_csv(metrics_tbl, 'output/03_pd_metrics.csv')
metrics_tbl

,metric,value
0,AUC,0.8639
1,Gini,0.7277
2,KS,0.5747
3,test_loans,45000.0000
4,pd_hat_mean,0.0077
5,pd_hat_p50,0.0024
6,pd_hat_p95,0.0325


**Reading the table:** AUC/Gini/KS measure how well the model ranks risky
loans above safe ones; higher is better. The calibration plot (saved to
`output/readme_assets/`) shows predicted and actual default rates lining up along
the diagonal -- the model is honest, not just discriminating.